In [1]:
import pandas as pd
import pickle

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

df = pd.read_csv("breast_cancer.csv")
df.columns = df.columns.str.strip()

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove rows with missing target values
df = df.dropna(subset=["Class"])

X = df.drop(columns=["Class"])
y = df["Class"].astype(int)

# Save feature names
columns = X.columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=2000))
])

model.fit(X_train, y_train)

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred, zero_division=0))
print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

# Save model and feature columns
with open("breast_cancer_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("breast_cancer_columns.pkl", "wb") as f:
    pickle.dump(columns, f)

print("\nModel trained and saved successfully!")

Accuracy: 0.9635036496350365

Classification Report:

              precision    recall  f1-score   support

           2       0.99      0.96      0.97        89
           4       0.92      0.98      0.95        48

    accuracy                           0.96       137
   macro avg       0.95      0.97      0.96       137
weighted avg       0.96      0.96      0.96       137


Confusion Matrix:

[[85  4]
 [ 1 47]]

Model trained and saved successfully!


In [2]:
import pandas as pd
import pickle
import gradio as gr

# Load saved model
with open("breast_cancer_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("breast_cancer_columns.pkl", "rb") as f:
    columns = pickle.load(f)

# Load dataset for default values
df = pd.read_csv("breast_cancer.csv")
df.columns = df.columns.str.strip()

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

defaults = df[columns].median(numeric_only=True).to_dict()

# Prediction function
def predict_breast_cancer(
    clump_thickness,
    uniformity_cell_size,
    uniformity_cell_shape,
    marginal_adhesion,
    single_epithelial_cell_size,
    bare_nuclei,
    bland_chromatin,
    normal_nucleoli,
    mitoses
):
    values = {
        "Clump Thickness": clump_thickness,
        "Uniformity of Cell Size": uniformity_cell_size,
        "Uniformity of Cell Shape": uniformity_cell_shape,
        "Marginal Adhesion": marginal_adhesion,
        "Single Epithelial Cell Size": single_epithelial_cell_size,
        "Bare Nuclei": bare_nuclei,
        "Bland Chromatin": bland_chromatin,
        "Normal Nucleoli": normal_nucleoli,
        "Mitoses": mitoses
    }

    input_df = pd.DataFrame([values])[columns]
    prediction = int(model.predict(input_df)[0])
    probabilities = model.predict_proba(input_df)[0]

    classes = model.named_steps["classifier"].classes_
    positive_index = list(classes).index(max(classes))
    probability = probabilities[positive_index] * 100

    result = (
        "Malignant (class {})".format(prediction)
        if prediction == max(classes)
        else "Benign (class {})".format(prediction)
    )

    return (
        f"Prediction: {result}\n"
        f"Probability for class {max(classes)}: {probability:.2f}%"
    )

# Create Gradio interface
inputs = [
    gr.Number(label="Clump Thickness", value=defaults.get("Clump Thickness", 5)),
    gr.Number(label="Uniformity of Cell Size", value=defaults.get("Uniformity of Cell Size", 4)),
    gr.Number(label="Uniformity of Cell Shape", value=defaults.get("Uniformity of Cell Shape", 4)),
    gr.Number(label="Marginal Adhesion", value=defaults.get("Marginal Adhesion", 3)),
    gr.Number(label="Single Epithelial Cell Size", value=defaults.get("Single Epithelial Cell Size", 3)),
    gr.Number(label="Bare Nuclei", value=defaults.get("Bare Nuclei", 1)),
    gr.Number(label="Bland Chromatin", value=defaults.get("Bland Chromatin", 3)),
    gr.Number(label="Normal Nucleoli", value=defaults.get("Normal Nucleoli", 3)),
    gr.Number(label="Mitoses", value=defaults.get("Mitoses", 1))
]

demo = gr.Interface(
    fn=predict_breast_cancer,
    inputs=inputs,
    outputs=gr.Textbox(label="Prediction Result"),
    title="Breast Cancer Prediction",
    description="Predict the dataset's benign or malignant class using Logistic Regression."
)

demo.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


In [ ]:
import pandas as pd
import pickle
from flask import Flask, request, render_template_string

# Load model and feature names
with open("breast_cancer_model.pkl", "rb") as f:
    model = pickle.load(f)

with open("breast_cancer_columns.pkl", "rb") as f:
    columns = pickle.load(f)

app = Flask(__name__)

# Default input values
df = pd.read_csv("breast_cancer.csv")
df.columns = df.columns.str.strip()

for col in df.columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

defaults = df[columns].median(numeric_only=True).to_dict()

labels = {
    "Clump Thickness": "Clump Thickness",
    "Uniformity of Cell Size": "Uniformity of Cell Size",
    "Uniformity of Cell Shape": "Uniformity of Cell Shape",
    "Marginal Adhesion": "Marginal Adhesion",
    "Single Epithelial Cell Size": "Single Epithelial Cell Size",
    "Bare Nuclei": "Bare Nuclei",
    "Bland Chromatin": "Bland Chromatin",
    "Normal Nucleoli": "Normal Nucleoli",
    "Mitoses": "Mitoses"
}

HTML = """
<!DOCTYPE html>
<html>
<head>
    <title>Breast Cancer Prediction</title>
    <style>
        body {
            font-family: Arial, sans-serif;
            background: #f2f5f9;
            padding: 25px;
        }
        .container {
            max-width: 800px;
            margin: auto;
            background: white;
            padding: 25px;
            border-radius: 10px;
            box-shadow: 0 3px 12px #ccc;
        }
        h1 {
            text-align: center;
            color: #173b63;
        }
        .grid {
            display: grid;
            grid-template-columns: 1fr 1fr;
            gap: 15px;
        }
        label {
            display: block;
            font-weight: bold;
            margin-bottom: 6px;
        }
        input {
            width: 100%;
            padding: 10px;
            box-sizing: border-box;
        }
        button {
            width: 100%;
            padding: 12px;
            margin-top: 20px;
            background: #1769aa;
            color: white;
            border: none;
            cursor: pointer;
        }
        .result {
            margin-top: 20px;
            padding: 15px;
            background: #e6f5e9;
            text-align: center;
            border-radius: 6px;
        }
        .error {
            color: red;
            margin-top: 15px;
        }
        @media(max-width: 600px) {
            .grid { grid-template-columns: 1fr; }
        }
    </style>
</head>
<body>
<div class="container">
    <h1>Breast Cancer Prediction</h1>

    <form method="POST">
        <div class="grid">
        {% for field in columns %}
            <div>
                <label>{{ labels[field] }}</label>
                <input type="number" name="{{ field }}" min="1" max="10"
                    step="1"
                    value="{{ values.get(field, defaults.get(field, 1)) }}"
                    required>
            </div>
        {% endfor %}
        </div>
        <button type="submit">Predict</button>
    </form>

    {% if result %}
    <div class="result">
        <h3>{{ result }}</h3>
        <p>Probability for malignant class: {{ probability }}%</p>
    </div>
    {% endif %}

    {% if error %}
    <p class="error">{{ error }}</p>
    {% endif %}
</div>
</body>
</html>
"""

@app.route("/", methods=["GET", "POST"])
def home():
    values = {}
    result = None
    probability = None
    error = None

    if request.method == "POST":
        try:
            values = request.form.to_dict()

            input_data = {
                col: float(values[col])
                for col in columns
            }

            input_df = pd.DataFrame([input_data], columns=columns)
            prediction = int(model.predict(input_df)[0])

            classes = model.named_steps["classifier"].classes_
            positive_class = max(classes)
            positive_index = list(classes).index(positive_class)
            probability = round(
                float(model.predict_proba(input_df)[0][positive_index]) * 100,
                2
            )

            result = (
                "Malignant"
                if prediction == positive_class
                else "Benign"
            )

        except Exception as e:
            error = str(e)

    return render_template_string(
        HTML,
        columns=columns,
        labels=labels,
        defaults=defaults,
        values=values,
        result=result,
        probability=probability,
        error=error
    )

if __name__ == "__main__":
    app.run(
        host="127.0.0.1",
        port=5004,
        debug=False,
        use_reloader=False
    )

# Code End


 * Serving Flask app '__main__'
 * Debug mode: off


 * Running on http://127.0.0.1:5004
Press CTRL+C to quit
127.0.0.1 - - [12/Oct/2026 00:57:26] "GET / HTTP/1.1" 200 -
127.0.0.1 - - [12/Oct/2026 00:57:30] "POST / HTTP/1.1" 200 -
127.0.0.1 - - [12/Oct/2026 00:57:50] "POST / HTTP/1.1" 200 -
